# Otimização de hiperparâmetros de modelos preditivos

- **Repositório:** `machine-learning-portfolio`  
- **Tecnologias:** Scikit-Learn  
- **Arquivo Original:** `Otimizador.ipynb`  

---


In [2]:
# Importar as bibliotecas necessárias
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col

# Criar a sessão do Spark
spark = SparkSession.builder.master("local").appName("Processar Excel").getOrCreate()

# Carregar o arquivo Excel com pandas, ignorando a primeira linha de cabeçalho
df_pandas = pd.read_excel('/content/SAI.xlsx', header=1, engine='openpyxl')

# Converter o pandas DataFrame para um PySpark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Verificar as primeiras linhas para garantir que as colunas foram carregadas corretamente
df_spark.show(5)

# Acessar diretamente a segunda coluna (índice 1)
df_with_parentheses = df_spark.withColumn(
    "extraido_parenteses",
    regexp_extract(col(df_spark.columns[1]), r"\((.*?)\)", 1)
)

# Coletar os valores extraídos da coluna "extraido_parenteses" em um vetor (lista)
resultado = df_with_parentheses.select("extraido_parenteses").rdd.flatMap(lambda x: x).collect()

# Criar uma lista para armazenar as palavras únicas
palavras_unicas = []

# Verificar se a palavra já foi adicionada, caso contrário, adicionar à lista
for item in resultado:
    if item not in palavras_unicas:
        palavras_unicas.append(item)

# Exibir as palavras únicas pulando linha
for item in palavras_unicas:
    print(item)

# Fechar a sessão do Spark
spark.stop()


+---+-------------------------------+---+
| F1|Digitação de Ordens de Crédito | PE|
+---+-------------------------------+---+
| F1|           Digitação de Orde...| AL|
| F1|           Digitação de Orde...| PE|
| F1|           Digitação de Orde...| AL|
| F1|           Digitação de Orde...| PE|
| F1|           Digitação de Orde...| AL|
+---+-------------------------------+---+
only showing top 5 rows

Incluir
Listar
Excluir
Detalhar
Alterar

PDF
Exportar Excel
Inserir
Todos
Ficha
Filtrados
Bookmark
Excel
Importar
Executar
Selecionar
Filtrar
Pesquisar
Conveniar
Exportar
Cancelar
Validar
Calcular
Excel 
Gerar


In [ ]:
# Importar as bibliotecas necessárias
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col

# Criar a sessão do Spark
spark = SparkSession.builder.master("local").appName("Processar Excel").getOrCreate()

# Carregar o arquivo Excel com pandas, ignorando a primeira linha de cabeçalho
df_pandas = pd.read_excel('/content/CCO.xlsx', header=1, engine='openpyxl')

# Converter o pandas DataFrame para um PySpark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Verificar as primeiras linhas para garantir que as colunas foram carregadas corretamente
df_spark.show(5)

# Acessar diretamente a segunda coluna (índice 1)
df_with_parentheses = df_spark.withColumn(
    "extraido_parenteses",
    regexp_extract(col(df_spark.columns[1]), r"\((.*?)\)", 1)
)

# Coletar os valores extraídos da coluna "extraido_parenteses" em um vetor (lista)
resultado = df_with_parentheses.select("extraido_parenteses").rdd.flatMap(lambda x: x).collect()

# Criar uma função para procurar a palavra exata e mostrar a linha
def encontrar_linha(palavra):
    linhas_encontradas = []
    for i, item in enumerate(resultado):
        if item == palavra:  # Compara a palavra exatamente, considerando maiúsculas e minúsculas
            linhas_encontradas.append(i + 2)  # +2 porque as linhas começam em 1 e estamos ignorando a primeira
    return linhas_encontradas

# Solicitar a entrada do usuário
palavra_busca = input("Digite a palavra exata para buscar dentro dos parênteses: ")

# Encontrar e exibir as linhas onde a palavra está presente
linhas = encontrar_linha(palavra_busca)

if linhas:
    print(f"A palavra '{palavra_busca}' foi encontrada nas seguintes linhas: {linhas}")
else:
    print(f"A palavra '{palavra_busca}' não foi encontrada em nenhum parêntese.")

# Fechar a sessão do Spark
spark.stop()


+---+----------------------------+---+
| F1|Inspetoria e Grupo (Inserir)| AL|
+---+----------------------------+---+
| F1|        Inspetoria e Grup...| AL|
| F1|        Inspetoria e Grup...| AL|
| F1|        Inspetoria e Grup...|NaN|
| F1|        Inspetoria e Grup...|NaN|
| F1|        Relatórios - Insp...|NaN|
+---+----------------------------+---+
only showing top 5 rows

Digite a palavra exata para buscar dentro dos parênteses: Filtar
A palavra 'Filtar' foi encontrada nas seguintes linhas: [450, 464, 498, 691, 695, 701, 704]


In [5]:
import pandas as pd

# Dicionário de mapeamento
mapeamento = {
    "Alterar": "AL",
    "Excluir": "AL",
    "Pesquisar": "PE",
    "Listar": "PE",
    "PDF": "AL",
    "Inserir": "AL",
    "Executar": "AL",
    "Selecionar": "PE",
    "Bloquear": "AL",
    "Excel": "AL",
    "Filtrar": "PE",
    "Enviar": "AL",
    "Desbloquear": "AL",
    "Emitir": "AL",
    "Vincular": "AL",
    "Cadastrar": "AL",
    "Importar": "AL",
    "Consultar": "PE",
    "Detalhar": "PE",
    "Simular": "PE",
    "Gerar": "AL",
    "Reabrir": "AL",
    "Inscrever": "AL",
    "Exportar": "AL",
    "TXT": "AL",
    "Incluir": "AL",
    "Conveniar": "PE",
    "Cancelar": 'PE',
    "Validar": "PE",
    "Calcular": "PE"
}

# 1. Carregar o arquivo Excel
arquivo = "/content/SAI.xlsx"  # Altere o caminho do arquivo se necessário
df_pandas = pd.read_excel(arquivo, header=0, engine='openpyxl')  # Lê o arquivo Excel

# 2. Extrair o texto entre parênteses da segunda coluna
# Vamos assumir que a segunda coluna tem o título 'Funcionalidade'

# Função para preencher a coluna 'Tipo' apenas se estiver vazia
def preencher_coluna_tipo_ou_pular(row):
    # Se a célula na coluna 'Tipo' estiver vazia (não tem valor), preenche com o mapeamento
    if pd.isna(row['Tipo']) or row['Tipo'] == "":
        # Extrai o valor entre parênteses da coluna 'Funcionalidade'
        palavra = row['Funcionalidade']
        # Usar expressao regular diretamente na string para extrair o valor entre parênteses
        palavra_extraida = pd.Series(palavra).str.extract(r'\((.*?)\)', expand=False).iloc[0]
        # Preenche a coluna 'Tipo' com o valor mapeado
        return mapeamento.get(palavra_extraida, "AL")  # Usa o mapeamento para determinar o valor
    else:
        # Caso já tenha algum valor, mantém o valor atual
        return row['Tipo']

# Aplicar a função para preencher a coluna 'Tipo' (a terceira coluna)
df_pandas['Tipo'] = df_pandas.apply(preencher_coluna_tipo_ou_pular, axis=1)

# 3. Salvar em um novo arquivo Excel
novo_arquivo = "/content/resultado_processado.xlsx"  # Altere o caminho do arquivo se necessário
df_pandas.to_excel(novo_arquivo, index=False, engine='openpyxl')

# Confirmar que o arquivo foi salvo com sucesso
print(f"O arquivo foi salvo em: {novo_arquivo}")


O arquivo foi salvo em: /content/resultado_processado.xlsx
